# NB_ingest_to_links — Silver → Links

Reads each Silver source table defined in the DV model config,
resolves FK hash keys for each hub reference, computes the composite
link hash key, and performs an INSERT-only MERGE into the vault link table.
Links are insert-only by DV 2.0 design.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
import uuid as _uuid
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("CATALOG",       "workspace",  "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA",  "vault",      "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver",     "Silver schema")
dbutils.widgets.text("MODEL_PATH",    "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS",  "",           "Optional: only process records >= this timestamp")
dbutils.widgets.text("ALERT_CHANNEL", "log",        "log | webhook | all")
dbutils.widgets.text("WEBHOOK_URL",   "",           "Optional Slack/Teams webhook")

CATALOG        = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA   = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA  = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH     = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS   = dbutils.widgets.get("WATERMARK_TS")
ALERT_CHANNEL  = dbutils.widgets.get("ALERT_CHANNEL")
WEBHOOK_URL    = dbutils.widgets.get("WEBHOOK_URL") or None
VAULT_RUN_ID   = str(_uuid.uuid4())

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['links'])} links from model")

In [ ]:
import uuid as _uuid
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")
VAULT_RUN_ID  = str(_uuid.uuid4())

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['links'])} links from model")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

link_counts = {}

for lnk_cfg in model['links']:
    if not lnk_cfg.get('enabled', True):
        print(f"  Skipping disabled link: {lnk_cfg['name']}")
        continue

    lnk_name    = lnk_cfg['name']
    source_tbl  = lnk_cfg['source_table']
    hub_refs    = lnk_cfg['hub_references']
    load_dt_col = lnk_cfg['load_date_column']
    rec_src     = lnk_cfg['record_source']
    lk_alias    = f"HK_{lnk_name[4:]}"
    target_tbl  = f"{CATALOG}.{VAULT_SCHEMA}.{lnk_name.lower()}"

    # Ensure target table exists
    create_link_table(lnk_cfg)

    # Read Silver source
    src_df = spark.table(source_tbl)
    if WATERMARK_TS:
        src_df = src_df.filter(F.col(load_dt_col) >= WATERMARK_TS)

    # Compute FK hash keys for each hub reference
    for ref in hub_refs:
        hub_n  = ref['hub']
        src_col = ref['source_column']
        hk_col  = f"HK_{hub_n[4:]}"
        src_df = src_df.withColumn(
            hk_col,
            F.sha2(F.concat_ws('||', F.upper(F.trim(F.col(src_col).cast('string')))), 256)
        )

    # Composite link hash key from all FK source columns
    fk_src_cols = [ref['source_column'] for ref in hub_refs]
    src_df = (
        src_df
        .withColumn(lk_alias, generate_hash_key(fk_src_cols))
        .withColumn('LOAD_DATE', F.col(load_dt_col).cast('timestamp'))
        .withColumn('RECORD_SOURCE', F.lit(rec_src))
    )

    # Select only the columns that exist in the target schema
    hk_cols = [f"HK_{ref['hub'][4:]}" for ref in hub_refs]
    src_df = src_df.select(lk_alias, *hk_cols, 'LOAD_DATE', 'RECORD_SOURCE').dropDuplicates([lk_alias])

    # INSERT-only MERGE
    lnk_tbl = DeltaTable.forName(spark, target_tbl)
    (
        lnk_tbl.alias('tgt')
        .merge(src_df.alias('src'), f"tgt.{lk_alias} = src.{lk_alias}")
        .whenNotMatchedInsertAll()
        .execute()
    )

    count = spark.table(target_tbl).count()
    link_counts[lnk_name] = count
    print(f"  {lnk_name}: {count:,} total rows in {target_tbl}")

print('\nLink ingestion complete.')

In [ ]:
# ---------------------------------------------------------------------------
# Task 3.1 — Link DQ: both hub FKs present (no null HK references)
# Task 4.3 — Alert on FAIL
# ---------------------------------------------------------------------------
dq_failures = []

for lnk_cfg in model['links']:
    if not lnk_cfg.get('enabled', True):
        continue

    lnk_name   = lnk_cfg['name']
    hub_refs   = lnk_cfg['hub_references']
    target_tbl = f"{CATALOG}.{VAULT_SCHEMA}.{lnk_name.lower()}"
    hk_cols    = [f"HK_{ref['hub'][4:]}" for ref in hub_refs]

    for hk_col in hk_cols:
        null_count = spark.sql(f"""
            SELECT SUM(CASE WHEN {hk_col} IS NULL THEN 1 ELSE 0 END) AS n FROM {target_tbl}
        """).collect()[0]["n"] or 0

        status = "PASS" if null_count == 0 else "FAIL"
        write_dq_result(
            spark, run_id=VAULT_RUN_ID, layer="vault", table_name=lnk_name.lower(),
            check_name=f"fk_present:{hk_col}", status=status,
            observed_value=float(null_count), threshold=0.0,
            message=f"{null_count} null {hk_col} in {lnk_name}" if null_count else None,
        )
        if status == "FAIL":
            alert_dq_failure(
                table_name=lnk_name.lower(), check_name=f"fk_present:{hk_col}",
                observed_value=null_count, layer="vault",
                alert_channel=ALERT_CHANNEL, webhook_url=WEBHOOK_URL,
            )
            dq_failures.append(f"{lnk_name}.{hk_col}: {null_count} null FK(s)")

print("Link ingestion complete.")
if dq_failures:
    raise RuntimeError("Link DQ FAIL:\n" + "\n".join(dq_failures))

print("Link DQ checks PASSED.")